# EDA On `combine_tracking.csv` Dataset


In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    RAW_DATA_DIR,
)

import polars as pl 
import pandas as pd 
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"

In [ ]:
# ==========================================
# Load Data
# ==========================================

combine_tracking_df = (
    pl.read_csv(
        str(RAW_DATA_DIR / "combine_tracking.csv"), 
    )
)

print("Combine Player Tracking Data:")
print(f"    - Dimensions: {combine_tracking_df.shape}")
combine_tracking_df

In [ ]:
# ========================================
# Missing Data
# ========================================

# No missing data

combine_tracking_df.select(pl.all().is_null().sum())

In [ ]:
# ========================================
# draft_year
# ========================================

combine_tracking_df.group_by("draft_year").agg(count = pl.len()).with_columns(perc = (100 * pl.col("count") / pl.sum("count")).round(2))

In [ ]:
# =======================================
# event_id
# =======================================

print(f"  - Number of Events During The Combine That We Have Tracking Data: {combine_tracking_df.select("event_id").n_unique()}")
print(f"    - Min Event Length (in tenths of seconds): {combine_tracking_df.group_by("event_id").agg(length = pl.len()).select(pl.min("length")).item()}")
print(f"    - Min Event Length (in tenths of seconds): {combine_tracking_df.group_by("event_id").agg(length = pl.len()).select(pl.max("length")).item()}")

In [ ]:
# =======================================
# event_id CONTINUED
# =======================================

# Look at the length of events

length_of_event_df = (
    combine_tracking_df
    .group_by("event_id")
    .agg(num_frames = pl.len())
)

print(f"  - Number of Events < 2.5 Seconds (25 rows): {length_of_event_df.filter(pl.col("num_frames") < 25).height}")
print(f"  - Number of Events > 20 Seconds (200 rows): {length_of_event_df.filter(pl.col("num_frames") > 200).height}")

fig = px.violin(
    length_of_event_df, 
    y = "num_frames", 
    points = "all", 
    title = "Num 0.1 Second Frames For Each Event"
)

fig.show()

In [ ]:
# =======================================
# event_id CONTINUED
# =======================================

# Do any players have multiple tries for the same drill?
#       YES, some players have MULTIPLE events for the same drill (based on drill name)

# NOTE: Variable attempts may not go in order 1, 2, 3, 4, ...
#       Someone may have 4 attempts in our dataset for a drill in our
#       dataset and the variable attempt has values 1, 2, 3, and 5.
#       Perhaps some attempts get thrown out, like attempt 4 in that instance

print("Max Number of Attempts a Player Has On One Specific Drill (drill_name):")
print(f"    - Number of Drills With Only 1 Attempt: {combine_tracking_df.group_by("nfl_id", "drill_name").agg(num_attempts = pl.col("event_id").n_unique()).filter(pl.col("num_attempts") == 1).height}/{combine_tracking_df.select("event_id").n_unique()}")
combine_tracking_df.group_by("nfl_id", "drill_name").agg(num_attempts = pl.col("event_id").n_unique()).filter(pl.col("num_attempts") > 1).sort("num_attempts", descending = True)

In [ ]:
# =======================================
# nfl_id
# =======================================

print(f"  - Number of Unique Player IDs: {combine_tracking_df.select("nfl_id").n_unique()}")

In [ ]:
# ======================================
# entity_type
# ======================================

combine_tracking_df.group_by("entity_type").agg(count = pl.len()).with_columns(perc = (100 * pl.col("count") / pl.sum("count")).round(2))

In [ ]:
# ======================================
# entity_type CONTINUED
# ======================================

# What kind of drills does ball data get used for?
#       Looks like below that the kind of drills that ball data gets 
#       used for is when players are running routes

print("Drills where ball tracking data is used:")
sorted(combine_tracking_df.filter(pl.col("entity_type") == "BALL").select("drill_name").unique().to_series().to_list())

In [ ]:
# =====================================
# drill_type (broad drill category)
# =====================================

print(f"  - Number of Unique Drill Types: {combine_tracking_df.select("drill_type").n_unique()}")
sorted(combine_tracking_df.select("drill_type").unique().to_series().to_list())

In [ ]:
# ===================================
# drill_type CONTINUED
# ===================================

print("Number of Events For Each Drill Type: ")
combine_tracking_df.group_by("drill_type").agg(num_events = pl.col("event_id").n_unique()).with_columns(perc_events = (100 *pl.col("num_events") / pl.sum("num_events")).round(2)).sort("num_events")

In [ ]:
# =====================================
# drill_name (specific drill name)
# =====================================

# NOTE: drill_name has MESSY DATA
#       EX: Has both "PASS_PRO_MIRROR_DRILL" and "PASS_PRO-MIRROR_DRILL"

print(f"  - Number of Unique Drill Types: {combine_tracking_df.select("drill_name").n_unique()}")
sorted(combine_tracking_df.select("drill_name").unique().to_series().to_list())

In [ ]:
# ======================================
# drill_type / drill_name
# ======================================

print("Number of Specific Drill Names (drill_name) For Each General Drill Type (drill_type):")
combine_tracking_df.group_by("drill_type").agg(num_drill_names = pl.col("drill_name").n_unique()).sort("num_drill_names")

In [ ]:
# ======================================
# drill_type / drill_name CONTINUED
# ======================================

print(f"SKILL_DRILLS_DB Events: {sorted(combine_tracking_df.filter(pl.col("drill_type") == "SKILL_DRILLS_DB").select("drill_name").unique().to_series().to_list())}")
print(f"SKILL_DRILLS_DL Events: {sorted(combine_tracking_df.filter(pl.col("drill_type") == "SKILL_DRILLS_DL").select("drill_name").unique().to_series().to_list())}")
print(f"SKILL_DRILLS_LB Events: {sorted(combine_tracking_df.filter(pl.col("drill_type") == "SKILL_DRILLS_LB").select("drill_name").unique().to_series().to_list())}")
print(f"SKILL_DRILLS_OL Events: {sorted(combine_tracking_df.filter(pl.col("drill_type") == "SKILL_DRILLS_OL").select("drill_name").unique().to_series().to_list())}")
print(f"SKILL_DRILLS_TE Events: {sorted(combine_tracking_df.filter(pl.col("drill_type") == "SKILL_DRILLS_TE").select("drill_name").unique().to_series().to_list())}")
print(f"SKILL_DRILLS_WR Events: {sorted(combine_tracking_df.filter(pl.col("drill_type") == "SKILL_DRILLS_WR").select("drill_name").unique().to_series().to_list())}")

In [ ]:
# ===========================================
# attempt
# ===========================================

print(f"  - Unique Values Of attempt (N = {combine_tracking_df.select("attempt").n_unique()}): {sorted(combine_tracking_df.select("attempt").unique().to_series().to_list())}")
print("")

print("  - Max Number of Attempts a Player Has On One Specific Drill (drill_name):")
print(f"    - Number of Drills With Only 1 Attempt: {combine_tracking_df.group_by("nfl_id", "drill_name").agg(num_attempts = pl.col("event_id").n_unique()).filter(pl.col("num_attempts") == 1).height}/{combine_tracking_df.select("event_id").n_unique()}")
combine_tracking_df.group_by("nfl_id", "drill_name").agg(num_attempts = pl.col("event_id").n_unique()).filter(pl.col("num_attempts") > 1).sort("num_attempts", descending = True)

In [ ]:
# ==================================================
# x
# ==================================================

# All values of x are within their expected range

print(f"  - Min/Max Values of x: [{min(combine_tracking_df["x"])}, {max(combine_tracking_df["x"])}]")

In [ ]:
# ==================================================
# y 
# ==================================================

print(f"  - Min/Max Values of y: [{min(combine_tracking_df["y"])}, {max(combine_tracking_df["y"])}]")
print(f"     - % of 10Hz values < 0:     {round(100 * combine_tracking_df.filter(pl.col("y") < 0).height / combine_tracking_df.height, 2)}%")
print(f"     - % of 10Hz values > 53.33: {round(100 * combine_tracking_df.filter(pl.col("y") > 53.333).height / combine_tracking_df.height, 2)}%")

In [ ]:
# =================================================
# s (speed)
# =================================================

max_speed_allowed = 13.69

print(f"  - Min/Max Values of s: [{min(combine_tracking_df["s"])}, {max(combine_tracking_df["s"])}]")

# Historical max speed among NFL players are 13.69 y/s
# NOTE: Many of these speed values are very unrealistic

# What proportion of speeds are > max allowed speed? 
print(f"  - Percentage of speed values > max allowed speed: {round(100 * combine_tracking_df.filter(pl.col("s") > max_speed_allowed).height / combine_tracking_df.height, 2)}%")

# How many events have speed > max allowed speed
print(f"  - Percentage of events that have a speed > max allowed speed: {combine_tracking_df.filter(pl.col("s") > max_speed_allowed).select("event_id").n_unique()}/{combine_tracking_df.select("event_id").n_unique()} ({round(100 * (combine_tracking_df.filter(pl.col("s") > max_speed_allowed).select("event_id").n_unique()) / (combine_tracking_df.select("event_id").n_unique()) , 5)}%)")


fig = px.violin(
    combine_tracking_df, 
    y = "s", 
    points = "all"
)

fig.show()

In [ ]:
# =================================================
# a (acceleration)
# =================================================

max_acceleration_allowed = 10

print(f"  - Min/Max Values of a: [{min(combine_tracking_df["a"])}, {max(combine_tracking_df["a"])}]")

# According to online resources, max values of acceleration are around 10 y/s^2
# NOTE: Many of these acceleration values are very unrealistic

# What proportion of acceleration are > max allowed acceleration? 
print(f"  - Percentage of acceleration values > max allowed acceleration: {round(100 * combine_tracking_df.filter(pl.col("a") > max_acceleration_allowed).height / combine_tracking_df.height, 2)}%")

# How many events have acceleration > max allowed acceleration?
print(f"  - Percentage of events that have a acceleration > max allowed acceleration: {combine_tracking_df.filter(pl.col("a") > max_acceleration_allowed).select("event_id").n_unique()}/{combine_tracking_df.select("event_id").n_unique()} ({round(100 * (combine_tracking_df.filter(pl.col("a") > max_acceleration_allowed).select("event_id").n_unique()) / (combine_tracking_df.select("event_id").n_unique()) , 5)}%)")

fig = px.violin(
    combine_tracking_df, 
    y = "a", 
    points = "all"
)

fig.show()

In [ ]:
# ==================================================
# dis
# ==================================================

max_distance_allowed = 2

print(f"  - Min/Max Values of dis: [{min(combine_tracking_df["dis"])}, {max(combine_tracking_df["dis"])}]")

# According to online resources, max values of dis are around 2 y
# NOTE: Many of these dis values are very unrealistic

# What proportion of dis are > max allowed distance? 
print(f"  - Percentage of dis values > max allowed distance: {round(100 * combine_tracking_df.filter(pl.col("dis") > max_distance_allowed).height / combine_tracking_df.height, 2)}%")

# How many events have dis > max allowed distance?
print(f"  - Percentage of events that have a dis > max allowed distance: {combine_tracking_df.filter(pl.col("dis") > max_distance_allowed).select("event_id").n_unique()}/{combine_tracking_df.select("event_id").n_unique()} ({round(100 * (combine_tracking_df.filter(pl.col("dis") > max_distance_allowed).select("event_id").n_unique()) / (combine_tracking_df.select("event_id").n_unique()) , 5)}%)")

fig = px.violin(
    combine_tracking_df, 
    y = "dis", 
    points = "all"
)

fig.show()


In [ ]:
# ================================================
# dis / speed
# ================================================

# Distance traveled in previous frame should be perfectly
# correlated with speed. At the 10Hz leve, speed (y/s) should 
# be the same as 10 * distance (y)

fig = px.scatter(
    combine_tracking_df,
    x="dis",
    y="s",
    title="Yards Traveled In Last 0.1 Second vs Speed (y/s)"
)

# Red line is for relationship that we should see (speed = 10 * distance)
fig.add_shape(
    type="line",
    x0=0,
    y0=0,
    x1=3,
    y1=30,
    xref="x",
    yref="y",
    line=dict(color="red", width=1, dash="dash"),
    layer="above",
    name="y = 10x",
)

fig.show()

In [ ]:
# ================================================
# dis / speed CONTINUED
# ================================================

# Distance traveled in previous frame should be perfectly
# correlated with speed. At the 10Hz leve, speed (y/s) should 
# be the same as 10 * distance (y)

# This plot only looks at time where speed and distance are realistic
# (filtered out unrealistic time)

fig = px.scatter(
    combine_tracking_df.filter(pl.col("s") < max_speed_allowed, pl.col("dis") < 2),
    x="dis",
    y="s",
    title="Yards Traveled In Last 0.1 Second vs Speed (y/s)"
)

# Red line is for relationship that we should see (speed = 10 * distance)
fig.add_shape(
    type="line",
    x0=0,
    y0=0,
    x1=3,
    y1=30,
    xref="x",
    yref="y",
    line=dict(color="red", width=1, dash="dash"),
    layer="above",
    name="y = 10x",
)

fig.show()

In [ ]:
# ========================================
# speed / acceleration / distance
# ========================================

print(f"  - Percentage of events that have a s > 13.69 y/s OR a > 10 y/s^2 OR dis > 2 y: {combine_tracking_df.filter((pl.col("s") > max_speed_allowed) | (pl.col("a") > max_acceleration_allowed) | (pl.col("dis") > max_distance_allowed)).select("event_id").n_unique()}/{combine_tracking_df.select("event_id").n_unique()} ({round(100 * (combine_tracking_df.filter((pl.col("s") > max_speed_allowed) | (pl.col("a") > max_acceleration_allowed) | (pl.col("dis") > max_distance_allowed)).select("event_id").n_unique()) / (combine_tracking_df.select("event_id").n_unique()) , 5)}%)")
print(f"  - Percentage of  rows  that have a s > 13.69 y/s OR a > 10 y/s^2 OR dis > 2 y: {combine_tracking_df.filter((pl.col("s") > max_speed_allowed) | (pl.col("a") > max_acceleration_allowed) | (pl.col("dis") > max_distance_allowed)).height}/{combine_tracking_df.height} ({round(100 * combine_tracking_df.filter((pl.col("s") > max_speed_allowed) | (pl.col("a") > max_acceleration_allowed) | (pl.col("dis") > max_distance_allowed)).height / combine_tracking_df.height, 2)}%)")

In [ ]:
# ==================================================
# dir
# ==================================================

print(f"  - Min/Max Values of dir: [{min(combine_tracking_df["dir"])}, {max(combine_tracking_df["dir"])}]")